# Combined 2024–2025 Discount Dataset (VIC Metro)

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

DATA_DIR = Path('.')
if not (DATA_DIR / 'all_catalogue_products.csv').exists():
    DATA_DIR = Path('ML/Price-Prediction/price_prediction_by_shivam')

file_2024 = DATA_DIR / 'all_catalogue_products2024.csv'
file_2025 = DATA_DIR / 'all_catalogue_products.csv'
if not file_2024.exists() or not file_2025.exists():
    raise FileNotFoundError('Both 2024 and 2025 catalogue CSV files are required.')

data_2024 = pd.read_csv(file_2024, encoding='utf-8-sig')
data_2025 = pd.read_csv(file_2025, encoding='utf-8-sig')
data_2024 = data_2024[data_2024['region'].str.upper().eq('VIC METRO')].copy()
data_2025 = data_2025[data_2025['region'].str.upper().eq('VIC METRO')].copy()
data_2024['source_year'] = 2024
data_2025['source_year'] = 2025
df = pd.concat([data_2024, data_2025], ignore_index=True)
print(f'2024 VIC rows: {len(data_2024):,}')
print(f'2025 VIC rows: {len(data_2025):,}')
print(f'Combined rows: {len(df):,}')
df.head()

2024 VIC rows: 4,675
2025 VIC rows: 15,195
Combined rows: 19,870


,retailer,region,catalogue_title,catalogue_start_date,catalogue_end_date,page_number,product_name,special_price,regular_price,save_amount,...,ocr_confidence,needs_review,review_reason,source_page_image,source_catalogue_url,price_source,price_evidence,evidence_count,verification_status,source_year
0,Coles,VIC METRO,"Coles Catalogue December 25 - 31, 2024 VIC METRO",2024-12-25,2025-01-01,1,"Pepsi, Solo or Schweppes Soft Drink or Schweppes",9.0,18.0,9.0,...,95.74,True,single_price_evidence,https://caau.syd1.cdn.digitaloceanspaces.com/w...,https://www.catalogueau.com/coles/#catalogue=c...,displayed_price,displayed_price,1,review,2024
1,Coles,VIC METRO,"Coles Catalogue December 25 - 31, 2024 VIC METRO",2024-12-25,2025-01-01,1,Pringles Potato Crisps 118g-134g,2.0,5.5,3.5,...,79.00,True,low_name_confidence_strict;single_price_evidence,https://caau.syd1.cdn.digitaloceanspaces.com/w...,https://www.catalogueau.com/coles/#catalogue=c...,displayed_price,displayed_price,1,review,2024
2,Coles,VIC METRO,"Coles Catalogue December 25 - 31, 2024 VIC METRO",2024-12-25,2025-01-01,1,Coles Classic Burgers or Sausages . 400g-550g ...,10.0,NaN,NaN,...,93.05,True,offer_labels_missing;save_amount_missing;regul...,https://caau.syd1.cdn.digitaloceanspaces.com/w...,https://www.catalogueau.com/coles/#catalogue=c...,displayed_price,displayed_price,1,review,2024
3,Coles,VIC METRO,"Coles Catalogue December 25 - 31, 2024 VIC METRO",2024-12-25,2025-01-01,2,Nivea Sun Protect & Moisture Sunscreen Spray S...,14.0,28.0,14.0,...,93.25,False,NaN,https://caau.syd1.cdn.digitaloceanspaces.com/w...,https://www.catalogueau.com/coles/#catalogue=c...,save_was_arithmetic,displayed_price;save_was_arithmetic,2,verified,2024
4,Coles,VIC METRO,"Coles Catalogue December 25 - 31, 2024 VIC METRO",2024-12-25,2025-01-01,2,Bondi Sands Aero 1 Hour Express Tanning Foam 2...,13.5,27.0,13.5,...,95.49,True,single_price_evidence,https://caau.syd1.cdn.digitaloceanspaces.com/w...,https://www.catalogueau.com/coles/#catalogue=c...,unit_price,unit_price,1,review,2024


## Clean and prepare weekly data

In [2]:
data = df.copy()
data['catalogue_start_date'] = pd.to_datetime(data['catalogue_start_date'], errors='coerce')
number_columns = ['special_price', 'regular_price', 'save_amount', 'discount_percent', 'ocr_confidence', 'evidence_count']
for column in number_columns:
    data[column] = pd.to_numeric(data[column], errors='coerce')

def clean_name(name):
    name = str(name).lower().replace('&', ' and ')
    name = re.sub(r'\bcoca[ -]?cola\b', 'coca cola', name)
    name = re.sub(r'\bsoft drink\b', '', name)
    name = re.sub(r'\bkilograms?\b', 'kg', name)
    name = re.sub(r'\bgrams?\b', 'g', name)
    name = re.sub(r'\bmillilit(?:re|er)s?\b', 'ml', name)
    name = re.sub(r'\blit(?:re|er)s?\b', 'l', name)
    name = re.sub(r'[^a-z0-9.]+', ' ', name)
    return re.sub(r'\s+', ' ', name).strip()

data['original_product_name'] = data['product_name'].fillna('').str.strip()
data['canonical_product_name'] = data['original_product_name'].map(clean_name)
data = data[data['canonical_product_name'].str.count(r'[a-z]') >= 3].copy()
product_numbers = pd.Series(pd.factorize(data['canonical_product_name'])[0], index=data.index)
data['product_id'] = 'P' + product_numbers.astype(str).str.zfill(5)
data['brand_key'] = data['canonical_product_name'].str.split().str[0]

calculated_regular = data['special_price'] + data['save_amount']
data['regular_price_candidate'] = data['regular_price'].fillna(calculated_regular)
data['regular_price_origin'] = np.select(
    [data['regular_price'].notna(), data['regular_price'].isna() & calculated_regular.notna()],
    ['observed', 'calculated'], default='missing')
trusted_source = data['price_source'].isin(['displayed_price', 'save_was_arithmetic'])
trusted_review = data['verification_status'].eq('verified') | data['evidence_count'].ge(2)
special_ok = data['special_price'].between(0.01, 2_000)
special_trustworthy = trusted_source & trusted_review & data['ocr_confidence'].ge(85) & special_ok
calculated_discount = 100 * (data['regular_price_candidate'] - data['special_price']) / data['regular_price_candidate']
regular_ok = (data['regular_price_candidate'].between(0.01, 2_000)
              & data['regular_price_candidate'].ge(data['special_price'])
              & calculated_discount.between(0, 80))
data['special_price_clean'] = data['special_price'].where(special_trustworthy)
data['regular_price_clean'] = data['regular_price_candidate'].where(special_trustworthy & regular_ok)
data['price_trustworthy'] = data['special_price_clean'].notna().astype('int8')
data['price_outlier'] = (~special_ok | (data['regular_price_candidate'].notna() & ~regular_ok)).astype('int8')

key = ['retailer', 'region', 'product_id', 'catalogue_start_date']
before = len(data)
data = (data.sort_values(['price_trustworthy', 'regular_price_clean', 'evidence_count', 'ocr_confidence'], ascending=False)
        .drop_duplicates(key).sort_values(key).reset_index(drop=True))
print(f'Valid products: {data.product_id.nunique():,}')
print(f'Duplicates removed: {before - len(data):,}')
print(f'Trustworthy prices: {data.price_trustworthy.sum():,}')

Valid products: 8,508
Duplicates removed: 159
Trustworthy prices: 9,365


In [3]:
weeks = pd.DataFrame({'catalogue_start_date': sorted(data['catalogue_start_date'].dropna().unique())})
weeks['week_index'] = range(len(weeks))
product_columns = ['retailer', 'region', 'product_id', 'original_product_name', 'canonical_product_name', 'brand_key', 'pack_size']
products = data.sort_values('price_trustworthy', ascending=False).drop_duplicates(['retailer', 'region', 'product_id'])[product_columns]
weekly = products.merge(weeks, how='cross')
observed_columns = ['retailer', 'region', 'product_id', 'catalogue_start_date', 'special_price',
                    'special_price_clean', 'regular_price_clean', 'regular_price_origin', 'promo_type',
                    'price_source', 'price_trustworthy', 'price_outlier']
weekly = weekly.merge(data[observed_columns], on=['retailer', 'region', 'product_id', 'catalogue_start_date'],
                      how='left', indicator=True)
weekly['catalogue_observed'] = weekly.pop('_merge').eq('both').astype('int8')
weekly['promo_type'] = weekly['promo_type'].fillna('none')
weekly['price_trustworthy'] = weekly['price_trustworthy'].fillna(0).astype('int8')
weekly['price_outlier'] = weekly['price_outlier'].fillna(0).astype('int8')

series_key = ['retailer', 'region', 'product_id']
weekly = weekly.sort_values(series_key + ['catalogue_start_date']).reset_index(drop=True)
group = weekly.groupby(series_key, sort=False)
last_regular_price = group['regular_price_clean'].ffill()
last_regular_week = weekly['week_index'].where(weekly['regular_price_clean'].notna()).groupby(
    [weekly[column] for column in series_key]).ffill()
weekly['weeks_since_regular_price'] = weekly['week_index'] - last_regular_week
can_fill = weekly['weeks_since_regular_price'].between(1, 8)
weekly['regular_price_filled'] = weekly['regular_price_clean'].fillna(last_regular_price.where(can_fill))
weekly['regular_price_source'] = np.select(
    [weekly['regular_price_clean'].notna() & weekly['regular_price_origin'].eq('observed'),
     weekly['regular_price_clean'].notna() & weekly['regular_price_origin'].eq('calculated'), can_fill],
    ['observed', 'calculated', 'forward_filled'], default='missing')
weekly['is_special'] = (weekly['catalogue_observed'].eq(1)
                        & weekly['promo_type'].isin(['half_price', 'save_amount', 'special'])).astype('int8')
weekly['effective_price'] = weekly['special_price_clean'].fillna(weekly['regular_price_filled'])
weekly['price_inferred'] = (weekly['special_price_clean'].isna() & weekly['regular_price_filled'].notna()).astype('int8')
calculated_weekly_discount = 100 * (weekly['regular_price_filled'] - weekly['special_price_clean']) / weekly['regular_price_filled']
valid_weekly_discount = calculated_weekly_discount.between(0, 80) & weekly['is_special'].eq(1)
weekly['discount_percent'] = calculated_weekly_discount.where(valid_weekly_discount)
weekly.loc[weekly['is_special'].eq(0), 'discount_percent'] = 0.0
weekly['discount_percent_trustworthy'] = (valid_weekly_discount & weekly['special_price_clean'].notna()
                                                & weekly['regular_price_clean'].notna()).astype('int8')
weekly['history_count'] = group['price_trustworthy'].cumsum()
weekly['cold_start_product'] = weekly['history_count'].lt(3).astype('int8')

group = weekly.groupby(series_key, sort=False)
for lag in [1, 2, 4]:
    weekly[f'price_lag_{lag}'] = group['effective_price'].shift(lag)
weekly['discount_lag_1'] = group['discount_percent'].shift(1)
weekly['avg_price_4w'] = group['effective_price'].transform(lambda x: x.shift(1).rolling(4, min_periods=1).mean())
weekly['avg_price_8w'] = group['effective_price'].transform(lambda x: x.shift(1).rolling(8, min_periods=1).mean())
weekly['special_frequency_4w'] = group['is_special'].transform(lambda x: x.shift(1).rolling(4, min_periods=1).mean())
weekly['special_frequency_8w'] = group['is_special'].transform(lambda x: x.shift(1).rolling(8, min_periods=1).mean())
last_special_week = weekly['week_index'].where(weekly['is_special'].eq(1)).groupby(
    [weekly[column] for column in series_key]).ffill()
weekly['weeks_since_last_special'] = weekly['week_index'] - last_special_week
weekly['week_of_year'] = weekly['catalogue_start_date'].dt.isocalendar().week.astype('int16')
weekly['month'] = weekly['catalogue_start_date'].dt.month.astype('int8')
weekly['quarter'] = weekly['catalogue_start_date'].dt.quarter.astype('int8')
weekly['season'] = weekly['month'].map({12:'summer', 1:'summer', 2:'summer', 3:'autumn', 4:'autumn',
                                         5:'autumn', 6:'winter', 7:'winter', 8:'winter', 9:'spring',
                                         10:'spring', 11:'spring'})
weekly['year'] = weekly['catalogue_start_date'].dt.year.astype('int16')

group = weekly.groupby(series_key, sort=False)
weekly['target_week'] = group['catalogue_start_date'].shift(-1)
weekly['target_is_special_next_week'] = group['is_special'].shift(-1)
weekly['target_discount_percent_next_week'] = group['discount_percent'].shift(-1)
target_discount_trustworthy = group['discount_percent_trustworthy'].shift(-1).eq(1)
weekly['target_discount_percent_next_week'] = weekly['target_discount_percent_next_week'].where(target_discount_trustworthy)
weekly['classification_eligible'] = weekly['target_is_special_next_week'].notna().astype('int8')
weekly['discount_regression_eligible'] = (weekly['target_is_special_next_week'].eq(1)
                                          & weekly['target_discount_percent_next_week'].notna()).astype('int8')

## Save and validate

In [4]:
keep = weekly['catalogue_observed'].eq(1) | weekly['effective_price'].notna()
final_columns = [
    'retailer', 'region', 'product_id', 'original_product_name', 'canonical_product_name', 'brand_key', 'pack_size',
    'catalogue_start_date', 'week_index', 'catalogue_observed', 'promo_type', 'special_price',
    'special_price_clean', 'regular_price_filled', 'regular_price_source', 'weeks_since_regular_price',
    'effective_price', 'price_inferred', 'price_trustworthy', 'price_outlier', 'is_special', 'discount_percent',
    'discount_percent_trustworthy', 'history_count', 'cold_start_product', 'price_lag_1', 'price_lag_2',
    'price_lag_4', 'discount_lag_1', 'avg_price_4w', 'avg_price_8w', 'special_frequency_4w',
    'special_frequency_8w', 'weeks_since_last_special', 'week_of_year', 'month', 'quarter', 'season', 'year',
    'target_week', 'target_is_special_next_week', 'target_discount_percent_next_week',
    'classification_eligible', 'discount_regression_eligible']
final = weekly.loc[keep, final_columns].copy()

assert final['region'].eq('VIC METRO').all()
assert not final.duplicated(['retailer', 'region', 'product_id', 'catalogue_start_date']).any()
assert final.loc[final['regular_price_source'].eq('forward_filled'), 'weeks_since_regular_price'].between(1, 8).all()
assert final['season'].notna().all()
assert final.loc[final['classification_eligible'].eq(1), 'target_is_special_next_week'].notna().all()
assert final.loc[final['discount_regression_eligible'].eq(1), 'target_discount_percent_next_week'].between(0, 80).all()

OUTPUT_FILE = DATA_DIR / 'discount_price_combined_2024_2025_vic.csv'
final.to_csv(OUTPUT_FILE, index=False)
print(f'Date range: {final.catalogue_start_date.min().date()} to {final.catalogue_start_date.max().date()}')
print(f'Final rows: {len(final):,}')
print(f'Products retained: {final.product_id.nunique():,}')
print(f'Classification rows: {final.classification_eligible.sum():,}')
special_examples = int(final.loc[final['classification_eligible'].eq(1), 'target_is_special_next_week'].sum())
print(f'Next-week specials: {special_examples:,}')
print(f'Discount regression rows: {final.discount_regression_eligible.sum():,}')
print(f'Saved: {OUTPUT_FILE}')
final.head()

Date range: 2024-09-11 to 2025-12-31
Final rows: 59,539
Products retained: 8,508
Classification rows: 58,748
Next-week specials: 5,430
Discount regression rows: 3,043
Saved: discount_price_combined_2024_2025_vic.csv


,retailer,region,product_id,original_product_name,canonical_product_name,brand_key,pack_size,catalogue_start_date,week_index,catalogue_observed,...,week_of_year,month,quarter,season,year,target_week,target_is_special_next_week,target_discount_percent_next_week,classification_eligible,discount_regression_eligible
15,Coles,VIC METRO,P00000,"Pepsi, Solo or Schweppes Soft Drink or Schweppes",pepsi solo or schweppes or schweppes,pepsi,NaN,2024-12-25,15,1,...,52,12,4,summer,2024,2025-01-01,0.0,NaN,1,0
69,Coles,VIC METRO,P00001,Pringles Potato Crisps 118g-134g,pringles potato crisps 118g 134g,pringles,118g; 134g,2024-09-11,0,1,...,37,9,3,spring,2024,2024-09-18,0.0,NaN,1,0
73,Coles,VIC METRO,P00001,Pringles Potato Crisps 118g-134g,pringles potato crisps 118g 134g,pringles,118g; 134g,2024-10-09,4,1,...,41,10,4,spring,2024,2024-10-16,0.0,NaN,1,0
78,Coles,VIC METRO,P00001,Pringles Potato Crisps 118g-134g,pringles potato crisps 118g 134g,pringles,118g; 134g,2024-11-13,9,1,...,46,11,4,spring,2024,2024-11-20,0.0,NaN,1,0
79,Coles,VIC METRO,P00001,Pringles Potato Crisps 118g-134g,pringles potato crisps 118g 134g,pringles,118g; 134g,2024-11-20,10,0,...,47,11,4,spring,2024,2024-11-27,0.0,NaN,1,0
